# Main training + Evaluation notebook

In [ ]:
# standard library imports
from pathlib import Path
import json
import os
import random
import time
import math
import warnings

# data / arrays
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

# plotting 
import matplotlib.pyplot as plt

# machine learning / metrics
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

# pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# notebook display
from IPython.display import display, Image

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# datasets
DATASETS_DIR = PROJECT_ROOT / "datasets"

CHOLEC80_ROOT = DATASETS_DIR / "Cholec80"
VIDEOS_DIR = CHOLEC80_ROOT / "videos"

SMOKE_ROOT = DATASETS_DIR / "Smoke_Cholec80"
SMOKE_MAP_DIR = SMOKE_ROOT / "cholec80_mappings"

# artifacts
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ART_COMMON = ARTIFACTS_DIR / "common"
ART_INDICES = ARTIFACTS_DIR / "indices"
ART_FEATURES = ARTIFACTS_DIR / "features"
ART_EMBEDDINGS = ARTIFACTS_DIR / "embeddings"   
ART_MODELS = ARTIFACTS_DIR / "models"
ART_RESULTS = ARTIFACTS_DIR / "results"
ART_FIGS = ARTIFACTS_DIR / "figures"
ART_LOGS = ARTIFACTS_DIR / "logs"

for d in [ARTIFACTS_DIR, ART_COMMON, ART_INDICES, ART_FEATURES, ART_EMBEDDINGS, ART_MODELS, ART_RESULTS, ART_FIGS, ART_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

# common files from preprocessing
VIDEO_INV_CSV = ART_COMMON / "video_inventory.csv"
SMOKE_SEG_CSV = ART_COMMON / "smoke_segment_inventory.csv"
FEATURE_CACHE_SUMMARY_CSV = ART_COMMON / "feature_cache_summary.csv"

assert VIDEO_INV_CSV.exists(), f"Missing file : {VIDEO_INV_CSV}"
assert SMOKE_SEG_CSV.exists(), f"Missing file : {SMOKE_SEG_CSV}"

# dataset availability
CHOLEC80_AVAILABLE = CHOLEC80_ROOT.exists() and VIDEOS_DIR.exists()

if not CHOLEC80_AVAILABLE:
    print("Cholec80 available  : False (dataset not found, continuing anyway)")
else:
    print("Cholec80 available  : True")

# seed
SEED = 67 # try 18, 42, 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Project root        :", PROJECT_ROOT)
print("Datasets folder        :", DATASETS_DIR)
print("Cholec80 folder        :", CHOLEC80_ROOT)
print("Videos folder       :", VIDEOS_DIR)
print("\nDevice              :", DEVICE)

In [ ]:
def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2))


def save_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def finalize_plot(title, xlabel, ylabel, out_path: Path, show=True):
    plt.title(title, pad=12, fontweight="bold")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    if show:
        plt.show()
    plt.close()


def save_bar(categories, values, title, xlabel, ylabel, out_path: Path, rotate=20, show_values=True):
    plt.figure(figsize=(10, 6), dpi=120)
    bars = plt.bar(categories, values, edgecolor="black", linewidth=0.8)

    if show_values:
        for b, v in zip(bars, values):
            plt.text(
                b.get_x() + b.get_width() / 2,
                b.get_height(),
                f"{v:.2f}" if isinstance(v, float) else f"{v}",
                ha="center",
                va="bottom",
                fontsize=10
            )

    plt.xticks(rotation=rotate, ha="right")
    finalize_plot(title, xlabel, ylabel, out_path, show=True)


def save_hist(values, title, xlabel, ylabel, out_path: Path, bins=30):
    plt.figure(figsize=(10, 6), dpi=120)
    plt.hist(values, bins=bins, edgecolor="black", linewidth=0.8)
    finalize_plot(title, xlabel, ylabel, out_path, show=True)


def save_line(x, y, title, xlabel, ylabel, out_path: Path, label=None):
    plt.figure(figsize=(10, 6), dpi=120)
    plt.plot(x, y, linewidth=2.0, label=label)
    if label is not None:
        plt.legend()
    finalize_plot(title, xlabel, ylabel, out_path, show=True)


def save_multi_line(curves, title, xlabel, ylabel, out_path: Path):
    plt.figure(figsize=(10, 6), dpi=120)
    for item in curves:
        plt.plot(item["x"], item["y"], linewidth=2.0, label=item["label"])
    plt.legend()
    finalize_plot(title, xlabel, ylabel, out_path, show=True)

## Stage 3 - Training setup and baseline inputs 

In [ ]:
# stage 3 (main experiment) config

# selected run from preprocessing
RUN_NAME = "w2p0s_l2p0s_s0p2s"

# all available run folders for horizon comparison
HORIZON_RUNS = [
    "w0p5s_l2p0s_s0p2s",
    "w1p0s_l2p0s_s0p2s",
    "w2p0s_l2p0s_s0p2s",
    "w3p0s_l2p0s_s0p2s"
]

# model training settings
EPOCHS_BASELINE = 6
EPOCHS_FUSION = 8
BATCH_SIZE = 256
NUM_WORKERS = 0   
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.20

# sequence / features
PHYS_DIM = 6

# threshold search
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)

# whether to allow rebuilding physics cache if missing
ALLOW_BUILD_PHYSICS_CACHE = True

# optional limit for quick debugging
DEBUG_SMALL_RUN = False
DEBUG_MAX_ROWS_PER_SPLIT = 30000   # only used if DEBUG_SMALL_RUN = True

# plotting folders for this run
RUN_FIG_DIR = ensure_dir(ART_FIGS / RUN_NAME)
RUN_MODEL_DIR = ensure_dir(ART_MODELS / RUN_NAME)
RUN_RESULT_DIR = ensure_dir(ART_RESULTS / RUN_NAME)

print("Selected run :", RUN_NAME)
print("Figure dir   :", RUN_FIG_DIR)
print("Model dir    :", RUN_MODEL_DIR)
print("Result dir   :", RUN_RESULT_DIR)

In [ ]:
# load the selected run artifacts
# if any of these are missing, fail early 

RUN_DIR = ART_INDICES / RUN_NAME
assert RUN_DIR.exists(), f"Missing run folder : {RUN_DIR}"

CONFIG_JSON = RUN_DIR / "config.json"
SPLITS_JSON = RUN_DIR / "splits.json"
CLASS_BALANCE_JSON = RUN_DIR / "class_balance.json"
CLIPS_CSV = RUN_DIR / "clips_index.csv"

for p in [CONFIG_JSON, SPLITS_JSON, CLASS_BALANCE_JSON, CLIPS_CSV]:
    assert p.exists(), f"Missing file : {p}"

run_config = json.loads(CONFIG_JSON.read_text())
split_info = json.loads(SPLITS_JSON.read_text())
class_balance = json.loads(CLASS_BALANCE_JSON.read_text())

videos_df = pd.read_csv(VIDEO_INV_CSV).sort_values("video_id").reset_index(drop=True)
smoke_segments_df = pd.read_csv(SMOKE_SEG_CSV)
clips_df = pd.read_csv(CLIPS_CSV)

if DEBUG_SMALL_RUN:
    clips_df = (
        clips_df.groupby("split", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), DEBUG_MAX_ROWS_PER_SPLIT), random_state=SEED))
        .reset_index(drop=True)
    )
    print("DEBUG_SMALL_RUN is ON. Using smaller sampled clips_df.")

print("Run config:")
display(pd.DataFrame([run_config]))

print("Split counts:")
display(pd.DataFrame([split_info["counts"]]))

print("Class balance:")
display(pd.DataFrame([{
    "run_name": RUN_NAME,
    "overall_rows": class_balance["overall"]["rows"],
    "overall_pos_rate": class_balance["overall"]["pos_rate"],
    "train_pos_rate": class_balance["train"]["pos_rate"],
    "val_pos_rate": class_balance["val"]["pos_rate"],
    "test_pos_rate": class_balance["test"]["pos_rate"]
}]))

print("clips_df shape :", clips_df.shape)
display(clips_df.head(5))

In [ ]:
# helper to find cached backbone features

def resolve_backbone_dir(backbone_name: str) -> Path:
    candidate_dirs = [
        ART_FEATURES / backbone_name,
        ART_EMBEDDINGS / backbone_name
    ]

    for d in candidate_dirs:
        if d.exists() and len(list(d.glob("*.npy"))) > 0:
            return d

    raise RuntimeError(
        f"Could not find cached features for backbone '{backbone_name}'. "
        f"Checked : {[str(x) for x in candidate_dirs]}"
    )


def load_backbone_meta(backbone_name: str):
    d = resolve_backbone_dir(backbone_name)
    meta_path = d / "meta.json"

    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
    else:
        # fallback if metadata is missing
        if backbone_name == "effb0":
            meta = {"embedding_dim": 1280}
        elif backbone_name == "convnext_tiny":
            meta = {"embedding_dim": 768}
        else:
            raise RuntimeError(f"No metadata rule for backbone : {backbone_name}")

    return d, meta


EFF_DIR, EFF_META = load_backbone_meta("effb0")
CONVNEXT_DIR, CONVNEXT_META = load_backbone_meta("convnext_tiny")

print("EfficientNet dir :", EFF_DIR)
print("EfficientNet dim :", EFF_META["embedding_dim"])
print("ConvNeXt dir     :", CONVNEXT_DIR)
print("ConvNeXt dim     :", CONVNEXT_META["embedding_dim"])

In [ ]:
# split summary plot

# clip count by split
split_counts = clips_df["split"].value_counts().reindex(["train", "val", "test"])
save_bar(
    categories=split_counts.index.tolist(),
    values=split_counts.values.tolist(),
    title=f"Clip count by split ({RUN_NAME})",
    xlabel="split",
    ylabel="number of clips",
    out_path=RUN_FIG_DIR / "clip_count_by_split.png",
    rotate=0,
    show_values=True
)

# target balance by split
target_split_df = (
    clips_df.groupby(["split", "target"])
    .size()
    .reset_index(name="count")
)

pivot = target_split_df.pivot(index="split", columns="target", values="count").fillna(0)
pivot = pivot.reindex(["train", "val", "test"])

plt.figure(figsize=(10, 6), dpi=120)
plt.bar(pivot.index, pivot.get(0, pd.Series([0]*len(pivot), index=pivot.index)), label="target=0", edgecolor="black")
plt.bar(
    pivot.index,
    pivot.get(1, pd.Series([0]*len(pivot), index=pivot.index)),
    bottom=pivot.get(0, pd.Series([0]*len(pivot), index=pivot.index)),
    label="target=1",
    edgecolor="black"
)
plt.legend()
finalize_plot(
    title=f"Forecasting target balance by split ({RUN_NAME})",
    xlabel="split",
    ylabel="number of clips",
    out_path=RUN_FIG_DIR / "target_balance_by_split.png",
    show=True
)

# positive rate by split
pos_rate_df = (
    clips_df.groupby("split")["target"]
    .mean()
    .reindex(["train", "val", "test"])
    .reset_index()
)
save_bar(
    categories=pos_rate_df["split"].tolist(),
    values=(pos_rate_df["target"] * 100).round(2).tolist(),
    title=f"Positive target rate by split ({RUN_NAME})",
    xlabel="split",
    ylabel="positive rate (%)",
    out_path=RUN_FIG_DIR / "positive_rate_by_split.png",
    rotate=0,
    show_values=True
)

In [ ]:
# turn smoke segments into event rows - measure whether model actually catches onset events

run_w_frames = int(run_config["w_frames"])
run_fps = float(run_config["fps"])
run_clip_len_frames = int(run_config["clip_len_frames"])

event_rows = []

smoke_only = smoke_segments_df[smoke_segments_df["label"] == 1].copy()
smoke_only = smoke_only.sort_values(["video_id", "frame_start"]).reset_index(drop=True)

for r in smoke_only.itertuples(index=False):
    event_rows.append({
        "video_id": r.video_id,
        "event_start": int(r.frame_start),
        "event_end": int(r.frame_end),
        "event_length_frames": int(r.frame_end - r.frame_start + 1),
        "event_length_seconds": float((r.frame_end - r.frame_start + 1) / run_fps)
    })

events_df = pd.DataFrame(event_rows)
save_df(events_df, RUN_RESULT_DIR / "events_df.csv")

print("events_df shape :", events_df.shape)
display(events_df.head(10))

save_hist(
    values=events_df["event_length_seconds"].values,
    title=f"Smoke event length distribution ({RUN_NAME})",
    xlabel="event length (seconds)",
    ylabel="count",
    out_path=RUN_FIG_DIR / "event_length_distribution.png",
    bins=40
)

## Stage 4 - Physics feature construction

In [ ]:
# frame-level features

PHYSICS_DIR = ensure_dir(ART_FEATURES / "physics")

PHYS_FEATURE_NAMES = [
    "gray_mean",
    "gray_std",
    "lap_var",
    "diff_energy",
    "bright_ratio",
    "edge_ratio"
]

def safe_video_capture(video_path: Path) -> cv2.VideoCapture:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot open video : {video_path}")
    return cap


def compute_frame_physics(gray_u8, prev_gray_u8=None):
    gray = gray_u8.astype(np.float32)

    gray_mean = float(gray.mean())
    gray_std = float(gray.std())

    lap = cv2.Laplacian(gray_u8, cv2.CV_64F)
    lap_var = float(lap.var())

    if prev_gray_u8 is None:
        diff_energy = 0.0
    else:
        diff = cv2.absdiff(gray_u8, prev_gray_u8).astype(np.float32)
        diff_energy = float(diff.mean())

    bright_ratio = float((gray_u8 > 220).mean())

    edges = cv2.Canny(gray_u8, 50, 150)
    edge_ratio = float((edges > 0).mean())

    return np.array([
        gray_mean,
        gray_std,
        lap_var,
        diff_energy,
        bright_ratio,
        edge_ratio
    ], dtype=np.float32)


def verify_physics_file(path: Path, expected_frames: int, expected_dim: int = PHYS_DIM):
    if not path.exists():
        return False, "missing"

    try:
        arr = np.load(path, mmap_mode="r")
    except Exception as e:
        return False, f"load_failed : {repr(e)}"

    if arr.shape != (expected_frames, expected_dim):
        return False, f"bad_shape : {arr.shape} expected {(expected_frames, expected_dim)}"

    return True, "ok"


def build_physics_cache_for_video(video_path: Path, out_path: Path, expected_frames: int):
    cap = safe_video_capture(video_path)
    arr = np.zeros((expected_frames, PHYS_DIM), dtype=np.float32)

    idx = 0
    prev_gray = None

    while idx < expected_frames:
        ok, frame = cap.read()
        if not ok or frame is None:
            cap.release()
            return False, f"frame_read_failed at {idx}"

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        arr[idx] = compute_frame_physics(gray, prev_gray)
        prev_gray = gray
        idx += 1

    cap.release()
    np.save(out_path, arr)
    return True, "ok"

In [ ]:
# verify existing physics caches or build them if needed

physics_summary_rows = []

for r in tqdm(videos_df.itertuples(index=False), total=len(videos_df), desc="Physics cache check/build"):
    vid = r.video_id
    nframes = int(r.frame_count)
    video_path = Path(r.path)
    out_path = PHYSICS_DIR / f"{vid}.npy"

    ok_existing, msg_existing = verify_physics_file(out_path, nframes, PHYS_DIM)

    if ok_existing:
        physics_summary_rows.append({
            "video_id": vid,
            "status": "valid_existing",
            "message": "ok"
        })
        continue

    if not CHOLEC80_AVAILABLE:
        physics_summary_rows.append({
            "video_id": vid,
            "status": "missing_and_not_built",
            "message": "Cholec80 not available"
        })
        continue

    if not ALLOW_BUILD_PHYSICS_CACHE:
        physics_summary_rows.append({
            "video_id": vid,
            "status": "missing_and_not_built",
            "message": "build not allowed"
        })
        continue

    ok_new, msg_new = build_physics_cache_for_video(video_path, out_path, nframes)

    physics_summary_rows.append({
        "video_id": vid,
        "status": "rebuilt" if ok_new else "failed",
        "message": msg_new
    })

physics_summary_df = pd.DataFrame(physics_summary_rows)
save_df(physics_summary_df, RUN_RESULT_DIR / "physics_cache_summary.csv")

display(physics_summary_df["status"].value_counts().rename_axis("status").reset_index(name="count"))

save_bar(
    categories=physics_summary_df["status"].value_counts().index.tolist(),
    values=physics_summary_df["status"].value_counts().values.tolist(),
    title=f"Physics cache status ({RUN_NAME})",
    xlabel="status",
    ylabel="count",
    out_path=RUN_FIG_DIR / "physics_cache_status.png",
    rotate=20,
    show_values=True
)

failed_physics = physics_summary_df[physics_summary_df["status"] == "failed"]
if len(failed_physics) > 0:
    display(failed_physics.head(20))

In [ ]:
# compute normalization stats using only training videos

train_video_ids = split_info["videos"]["train"]
val_video_ids = split_info["videos"]["val"]
test_video_ids = split_info["videos"]["test"]

def compute_physics_norm_stats(video_ids, physics_dir: Path):
    total_n = 0
    sum_vec = np.zeros((PHYS_DIM,), dtype=np.float64)
    sum_sq_vec = np.zeros((PHYS_DIM,), dtype=np.float64)

    for vid in tqdm(video_ids, desc="Computing physics normalization"):
        p = physics_dir / f"{vid}.npy"
        ok, msg = verify_physics_file(p, expected_frames=int(videos_df.loc[videos_df["video_id"] == vid, "frame_count"].iloc[0]), expected_dim=PHYS_DIM)
        if not ok:
            raise RuntimeError(f"Physics cache invalid for {vid} : {msg}")

        arr = np.load(p, mmap_mode="r")
        x = np.asarray(arr, dtype=np.float64)

        total_n += x.shape[0]
        sum_vec += x.sum(axis=0)
        sum_sq_vec += (x ** 2).sum(axis=0)

    mean = sum_vec / total_n
    var = (sum_sq_vec / total_n) - (mean ** 2)
    std = np.sqrt(np.maximum(var, 1e-8))

    return mean.astype(np.float32), std.astype(np.float32)

physics_norm_path = RUN_RESULT_DIR / "physics_norm_stats.json"

if physics_norm_path.exists():
    physics_norm = json.loads(physics_norm_path.read_text())
    phys_mean = np.array(physics_norm["mean"], dtype=np.float32)
    phys_std = np.array(physics_norm["std"], dtype=np.float32)
else:
    phys_mean, phys_std = compute_physics_norm_stats(train_video_ids, PHYSICS_DIR)
    physics_norm = {
        "feature_names": PHYS_FEATURE_NAMES,
        "mean": phys_mean.tolist(),
        "std": phys_std.tolist()
    }
    save_json(physics_norm, physics_norm_path)

print("Physics normalization stats:")
display(pd.DataFrame({
    "feature": PHYS_FEATURE_NAMES,
    "mean": phys_mean,
    "std": phys_std
}))

## Stage 5 - Temporal model, dataloaders and training loop

In [ ]:
# dataset class for loading temporal clip sequences

class ClipSequenceDataset(Dataset):
    def __init__(
        self,
        clips_df: pd.DataFrame,
        videos_df: pd.DataFrame,
        backbone_dir: Path,
        embedding_dim: int,
        physics_dir: Path,
        phys_mean: np.ndarray,
        phys_std: np.ndarray,
        use_physics: bool
    ):
        self.df = clips_df.reset_index(drop=True).copy()
        self.backbone_dir = backbone_dir
        self.embedding_dim = embedding_dim
        self.physics_dir = physics_dir
        self.use_physics = use_physics
        self.phys_mean = phys_mean.astype(np.float32)
        self.phys_std = phys_std.astype(np.float32)

        self.frame_count_map = dict(zip(videos_df["video_id"], videos_df["frame_count"].astype(int)))

        # keep memmap handles open
        self.embed_cache = {}
        self.phys_cache = {}

        for vid in self.df["video_id"].unique().tolist():
            ep = self.backbone_dir / f"{vid}.npy"
            if not ep.exists():
                raise RuntimeError(f"Missing embedding file : {ep}")
            self.embed_cache[vid] = np.load(ep, mmap_mode="r")

            pp = self.physics_dir / f"{vid}.npy"
            if pp.exists():
                self.phys_cache[vid] = np.load(pp, mmap_mode="r")
            else:
                self.phys_cache[vid] = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        vid = row["video_id"]
        s = int(row["clip_start"])
        e = int(row["clip_end"])
        y = float(row["target"])
        t = int(row["t"])

        emb = np.array(self.embed_cache[vid][s:e+1], dtype=np.float32, copy=True)
        
        if emb.shape[0] != (e - s + 1):
            raise RuntimeError(f"Bad embedding slice shape for {vid} {s}:{e}")

        if self.use_physics:
            phys_arr = self.phys_cache[vid]
            if phys_arr is None:
                raise RuntimeError(f"Physics cache missing for {vid}, but use_physics=True")
            phys = np.array(phys_arr[s:e+1], dtype=np.float32, copy=True)
            phys = (phys - self.phys_mean) / self.phys_std
        else:
            phys = np.zeros((emb.shape[0], PHYS_DIM), dtype=np.float32)

        return {
            "x_emb": torch.from_numpy(emb),
            "x_phys": torch.from_numpy(phys),
            "y": torch.tensor(y, dtype=torch.float32),
            "video_id": vid,
            "t": torch.tensor(t, dtype=torch.long),
            "clip_start": torch.tensor(s, dtype=torch.long),
            "clip_end": torch.tensor(e, dtype=torch.long)
        }

In [ ]:
def make_split_dfs(clips_df):
    train_df = clips_df[clips_df["split"] == "train"].copy().reset_index(drop=True)
    val_df = clips_df[clips_df["split"] == "val"].copy().reset_index(drop=True)
    test_df = clips_df[clips_df["split"] == "test"].copy().reset_index(drop=True)

    return train_df, val_df, test_df


def make_loaders(
    clips_df,
    videos_df,
    backbone_name: str,
    use_physics: bool,
    batch_size: int = BATCH_SIZE
):
    if backbone_name == "effb0":
        backbone_dir = EFF_DIR
        embedding_dim = int(EFF_META["embedding_dim"])
    elif backbone_name == "convnext_tiny":
        backbone_dir = CONVNEXT_DIR
        embedding_dim = int(CONVNEXT_META["embedding_dim"])
    else:
        raise RuntimeError(f"Unknown backbone : {backbone_name}")

    train_df, val_df, test_df = make_split_dfs(clips_df)

    train_ds = ClipSequenceDataset(
        clips_df=train_df,
        videos_df=videos_df,
        backbone_dir=backbone_dir,
        embedding_dim=embedding_dim,
        physics_dir=PHYSICS_DIR,
        phys_mean=phys_mean,
        phys_std=phys_std,
        use_physics=use_physics
    )

    val_ds = ClipSequenceDataset(
        clips_df=val_df,
        videos_df=videos_df,
        backbone_dir=backbone_dir,
        embedding_dim=embedding_dim,
        physics_dir=PHYSICS_DIR,
        phys_mean=phys_mean,
        phys_std=phys_std,
        use_physics=use_physics
    )

    test_ds = ClipSequenceDataset(
        clips_df=test_df,
        videos_df=videos_df,
        backbone_dir=backbone_dir,
        embedding_dim=embedding_dim,
        physics_dir=PHYSICS_DIR,
        phys_mean=phys_mean,
        phys_std=phys_std,
        use_physics=use_physics
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    return train_df, val_df, test_df, train_loader, val_loader, test_loader, embedding_dim

In [ ]:
# temporal models

class TCNBlock(nn.Module):
    def __init__(self, channels, dilation, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x : [B,C,T]
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        return x + residual


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        # x : [B,T,D]
        attn = self.score(x).squeeze(-1)   # [B,T]
        attn = torch.softmax(attn, dim=1)
        pooled = torch.sum(x * attn.unsqueeze(-1), dim=1)  # [B,D]
        return pooled, attn


class EmbeddingOnlyTCN(nn.Module):
    def __init__(self, emb_dim, hidden_dim=192, dropout=0.2):
        super().__init__()

        self.emb_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.tcn = nn.Sequential(
            TCNBlock(hidden_dim, dilation=1, dropout=dropout),
            TCNBlock(hidden_dim, dilation=2, dropout=dropout),
            TCNBlock(hidden_dim, dilation=4, dropout=dropout)
        )

        self.pool = TemporalAttentionPool(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_emb, x_phys=None):
        # x_emb : [B,T,D]
        x = self.emb_proj(x_emb)           # [B,T,H]
        x = x.transpose(1, 2)              # [B,H,T]
        x = self.tcn(x)
        x = x.transpose(1, 2)              # [B,T,H]
        pooled, attn = self.pool(x)
        logits = self.head(pooled).squeeze(-1)
        return logits, attn


class PhysicsOnlyTCN(nn.Module):
    def __init__(self, phys_dim=PHYS_DIM, hidden_dim=64, dropout=0.2):
        super().__init__()

        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.tcn = nn.Sequential(
            TCNBlock(hidden_dim, dilation=1, dropout=dropout),
            TCNBlock(hidden_dim, dilation=2, dropout=dropout),
            TCNBlock(hidden_dim, dilation=4, dropout=dropout)
        )

        self.pool = TemporalAttentionPool(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x_emb, x_phys):
        x = self.phys_proj(x_phys)
        x = x.transpose(1, 2)
        x = self.tcn(x)
        x = x.transpose(1, 2)
        pooled, attn = self.pool(x)
        logits = self.head(pooled).squeeze(-1)
        return logits, attn


class FusionTCN(nn.Module):
    def __init__(self, emb_dim, phys_dim=PHYS_DIM, hidden_dim=192, phys_hidden=48, dropout=0.2):
        super().__init__()
        self.emb_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, phys_hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fusion_proj = nn.Sequential(
            nn.Linear(hidden_dim + phys_hidden, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        fusion_dim = hidden_dim

        self.tcn = nn.Sequential(
            TCNBlock(fusion_dim, dilation=1, dropout=dropout),
            TCNBlock(fusion_dim, dilation=2, dropout=dropout),
            TCNBlock(fusion_dim, dilation=4, dropout=dropout)
        )

        self.pool = TemporalAttentionPool(fusion_dim)

        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 96),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(96, 1)
        )

    def forward(self, x_emb, x_phys):
        emb = self.emb_proj(x_emb)      # [B,T,H]
        phys = self.phys_proj(x_phys)   # [B,T,P]
    
        fused = torch.cat([emb, phys], dim=-1)   # [B,T,H+P]
        fused = self.fusion_proj(fused)          # [B,T,H]
    
        x = fused.transpose(1, 2)                # [B,H,T]
        x = self.tcn(x)
        x = x.transpose(1, 2)                    # [B,T,H]
    
        pooled, attn = self.pool(x)
        logits = self.head(pooled).squeeze(-1)
        return logits, attn

class GatedFusionTCN(nn.Module):
    def __init__(
        self,
        emb_dim: int,
        phys_dim: int = PHYS_DIM,
        hidden_dim: int = 192,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()

        # project both modalities into the same hidden size
        self.emb_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # learn per dimension gate values in [0, 1]
        self.gate_net = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),
        )

        self.fusion_refine = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.tcn = nn.Sequential(
            TCNBlock(hidden_dim, dilation=1, dropout=dropout),
            TCNBlock(hidden_dim, dilation=2, dropout=dropout),
            TCNBlock(hidden_dim, dilation=4, dropout=dropout),
        )

        self.pool = TemporalAttentionPool(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 96),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(96, 1),
        )

    def forward(self, x_emb: torch.Tensor, x_phys: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # x_emb: [B, T, emb_dim]
        # x_phys: [B, T, phys_dim]

        emb = self.emb_proj(x_emb)      # [B, T, H]
        phys = self.phys_proj(x_phys)   # [B, T, H]

        gate_input = torch.cat([emb, phys], dim=-1)   # [B, T, 2H]
        gate = self.gate_net(gate_input)              # [B, T, H]

        fused = gate * emb + (1.0 - gate) * phys      # [B, T, H]
        fused = self.fusion_refine(fused)

        x = self.tcn(fused.transpose(1, 2)).transpose(1, 2)
        pooled, attn = self.pool(x)
        logits = self.head(pooled).squeeze(-1)
        return logits, attn

class FusionMeanPool(nn.Module):
    def __init__(self, emb_dim, phys_dim=PHYS_DIM, hidden_dim=192, phys_hidden=48, dropout=0.2):
        super().__init__()

        self.emb_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, phys_hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fusion_proj = nn.Sequential(
            nn.Linear(hidden_dim + phys_hidden, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 96),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(96, 1)
        )

    def forward(self, x_emb, x_phys):
        # x_emb : [B,T,D]
        # x_phys: [B,T,P]

        emb = self.emb_proj(x_emb)       # [B,T,H]
        phys = self.phys_proj(x_phys)    # [B,T,P]
        fused = torch.cat([emb, phys], dim=-1)
        fused = self.fusion_proj(fused)  # [B,T,H]

        pooled = fused.mean(dim=1)       # [B,H]

        # fake uniform attention so downstream code stays compatible
        B, T, _ = fused.shape
        attn = torch.ones((B, T), device=fused.device, dtype=fused.dtype) / T

        logits = self.head(pooled).squeeze(-1)
        return logits, attn


class FusionGRU(nn.Module):
    def __init__(self, emb_dim, phys_dim=PHYS_DIM, hidden_dim=192, phys_hidden=48, dropout=0.2):
        super().__init__()

        self.emb_proj = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, phys_hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.fusion_proj = nn.Sequential(
            nn.Linear(hidden_dim + phys_hidden, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.pool = TemporalAttentionPool(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 96),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(96, 1)
        )

    def forward(self, x_emb, x_phys):
        # x_emb : [B,T,D]
        # x_phys: [B,T,P]

        emb = self.emb_proj(x_emb)       # [B,T,H]
        phys = self.phys_proj(x_phys)    # [B,T,P]
        fused = torch.cat([emb, phys], dim=-1)
        fused = self.fusion_proj(fused)  # [B,T,H]

        x, _ = self.gru(fused)           # [B,T,H]
        pooled, attn = self.pool(x)      # [B,H], [B,T]

        logits = self.head(pooled).squeeze(-1)
        return logits, attn

In [ ]:
def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def get_pos_weight(train_df):
    pos = float(train_df["target"].sum())
    neg = float(len(train_df) - pos)

    if pos <= 0:
        return 1.0

    return max(1.0, neg / pos)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    n_items = 0

    for batch in tqdm(loader, desc="train", leave=False):
        x_emb = batch["x_emb"].to(device)
        x_phys = batch["x_phys"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()

        logits, _ = model(x_emb, x_phys)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += float(loss.item()) * len(y)
        n_items += len(y)

    return running_loss / max(n_items, 1)


@torch.no_grad()
def predict_loader(model, loader, device):
    model.eval()

    rows = []

    for batch in tqdm(loader, desc="predict", leave=False):
        x_emb = batch["x_emb"].to(device)
        x_phys = batch["x_phys"].to(device)
        y = batch["y"].detach().cpu().numpy()
        t = batch["t"].detach().cpu().numpy()
        clip_start = batch["clip_start"].detach().cpu().numpy()
        clip_end = batch["clip_end"].detach().cpu().numpy()
        video_ids = list(batch["video_id"])

        logits, attn = model(x_emb, x_phys)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        for i in range(len(video_ids)):
            rows.append({
                "video_id": video_ids[i],
                "t": int(t[i]),
                "clip_start": int(clip_start[i]),
                "clip_end": int(clip_end[i]),
                "target": int(y[i]),
                "prob": float(probs[i])
            })

    return pd.DataFrame(rows)


def binary_metrics_from_df(pred_df: pd.DataFrame, threshold: float):
    y_true = pred_df["target"].astype(int).values
    probs = pred_df["prob"].astype(float).values
    y_pred = (probs >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    try:
        pr_auc = average_precision_score(y_true, probs)
    except Exception:
        pr_auc = float("nan")

    try:
        roc_auc = roc_auc_score(y_true, probs)
    except Exception:
        roc_auc = float("nan")

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": float(threshold),
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }


def find_best_threshold(pred_df: pd.DataFrame, thresholds=THRESHOLDS):
    rows = []

    for th in thresholds:
        m = binary_metrics_from_df(pred_df, float(th))
        rows.append(m)

    th_df = pd.DataFrame(rows).sort_values(["f1", "precision", "recall"], ascending=False).reset_index(drop=True)
    best = th_df.iloc[0].to_dict()

    return best, th_df

In [ ]:
def compute_event_metrics(pred_df, events_df, video_inventory_df, threshold, w_frames, fps):
    pred_df = pred_df.copy()
    pred_df["pred"] = (pred_df["prob"] >= threshold).astype(int)

    # false alarms per minute
    total_minutes = float(video_inventory_df["duration_sec"].sum() / 60.0)

    pred_df = pred_df.sort_values(["video_id", "t"]).copy()
    pred_df["is_fp"] = ((pred_df["pred"] == 1) & (pred_df["target"] == 0)).astype(np.int8)

    stride_by_video = (
        pred_df.groupby("video_id")["t"]
        .apply(
            lambda s: int(np.min(np.diff(np.sort(s.drop_duplicates().to_numpy()))))
            if s.drop_duplicates().shape[0] >= 2 else 1
        )
        .to_dict()
    )

    pred_df["inferred_stride"] = pred_df["video_id"].map(stride_by_video).astype(int)
    pred_df["prev_fp"] = pred_df.groupby("video_id")["is_fp"].shift(fill_value=0)
    pred_df["prev_t"] = pred_df.groupby("video_id")["t"].shift()

    pred_df["new_fp_episode"] = (
        (pred_df["is_fp"] == 1)
        & (
            (pred_df["prev_fp"] == 0)
            | (pred_df["prev_t"].isna())
            | (pred_df["t"] != pred_df["prev_t"] + pred_df["inferred_stride"])
        )
    )

    fp_alarm_episodes = int(pred_df["new_fp_episode"].sum())
    far_per_min = fp_alarm_episodes / max(total_minutes, 1e-8)

    # clip prediction stability
    stability_rows = []
    for vid, g in pred_df.sort_values(["video_id", "t"]).groupby("video_id"):
        arr = g["pred"].values.astype(int)
        if len(arr) <= 1:
            stability = 1.0
            flips = 0
        else:
            flips = int(np.sum(arr[1:] != arr[:-1]))
            flip_rate = flips / (len(arr) - 1)
            stability = 1.0 - flip_rate

        stability_rows.append({
            "video_id": vid,
            "num_preds": int(len(arr)),
            "flips": int(flips),
            "stability": float(stability)
        })

    stability_df = pd.DataFrame(stability_rows)
    clip_prediction_stability = float(stability_df["stability"].mean()) if len(stability_df) > 0 else float("nan")

    # event detection / lead time
    event_eval_rows = []

    pred_by_video = {
        vid: g.sort_values("t").reset_index(drop=True)
        for vid, g in pred_df.groupby("video_id")
    }

    for r in events_df.itertuples(index=False):
        vid = r.video_id
        event_start = int(r.event_start)

        if vid not in pred_by_video:
            event_eval_rows.append({
                "video_id": vid,
                "event_start": event_start,
                "detected": 0,
                "lead_time_sec": np.nan
            })
            continue

        g = pred_by_video[vid]

        # valid forecast clips - clips ending before event start n still within the forecast window
        valid = g[
            (g["t"] < event_start) &
            ((event_start - g["t"]) <= w_frames) &
            (g["pred"] == 1)
        ].copy()

        if len(valid) == 0:
            event_eval_rows.append({
                "video_id": vid,
                "event_start": event_start,
                "detected": 0,
                "lead_time_sec": np.nan
            })
        else:
            earliest_valid = valid.sort_values("t").iloc[0]
            lead_frames = event_start - int(earliest_valid["t"])
            lead_time_sec = float(lead_frames / fps)

            event_eval_rows.append({
                "video_id": vid,
                "event_start": event_start,
                "detected": 1,
                "lead_time_sec": lead_time_sec
            })

    event_eval_df = pd.DataFrame(event_eval_rows)

    total_events = int(len(event_eval_df))
    detected_events = int(event_eval_df["detected"].sum())
    missed_events = int(total_events - detected_events)
    mdr = missed_events / max(total_events, 1)

    lead_times = event_eval_df.loc[event_eval_df["detected"] == 1, "lead_time_sec"].dropna().values

    out = {
        "threshold": float(threshold),
        "total_events": total_events,
        "detected_events": detected_events,
        "missed_events": missed_events,
        "event_detection_rate": float(detected_events / max(total_events, 1)),
        "missed_detection_rate": float(mdr),
        "mean_lead_time_sec": float(np.mean(lead_times)) if len(lead_times) > 0 else float("nan"),
        "median_lead_time_sec": float(np.median(lead_times)) if len(lead_times) > 0 else float("nan"),
        "false_alarms_per_min": float(far_per_min),
        "clip_prediction_stability": float(clip_prediction_stability)
    }

    return out, event_eval_df, stability_df

In [ ]:
def find_best_threshold_with_event_metrics(
    pred_df: pd.DataFrame,
    events_df: pd.DataFrame,
    video_inventory_df: pd.DataFrame,
    thresholds,
    w_frames: int,
    fps: float,
    strategy: str = "f1_event_balanced",
    max_false_alarms_per_min: float | None = None
):
    rows = []

    for th in thresholds:
        clip_m = binary_metrics_from_df(pred_df, float(th))
        event_m, _, _ = compute_event_metrics(
            pred_df=pred_df,
            events_df=events_df,
            video_inventory_df=video_inventory_df,
            threshold=float(th),
            w_frames=w_frames,
            fps=fps
        )

        row = {
            "threshold": float(th),
            "pr_auc": clip_m["pr_auc"],
            "roc_auc": clip_m["roc_auc"],
            "precision": clip_m["precision"],
            "recall": clip_m["recall"],
            "f1": clip_m["f1"],
            "tp": clip_m["tp"],
            "fp": clip_m["fp"],
            "tn": clip_m["tn"],
            "fn": clip_m["fn"],
            "event_detection_rate": event_m["event_detection_rate"],
            "missed_detection_rate": event_m["missed_detection_rate"],
            "mean_lead_time_sec": event_m["mean_lead_time_sec"],
            "median_lead_time_sec": event_m["median_lead_time_sec"],
            "false_alarms_per_min": event_m["false_alarms_per_min"],
            "clip_prediction_stability": event_m["clip_prediction_stability"],
        }

        if strategy == "f1_only":
            row["selection_score"] = row["f1"]

        elif strategy == "event_constrained":
            if max_false_alarms_per_min is not None and row["false_alarms_per_min"] > max_false_alarms_per_min:
                row["selection_score"] = -1e9
            else:
                row["selection_score"] = (
                    0.6 * row["event_detection_rate"] +
                    0.4 * row["f1"]
                )

        elif strategy == "f1_event_balanced":
            row["selection_score"] = (
                0.45 * row["f1"] +
                0.35 * row["event_detection_rate"] -
                0.20 * row["false_alarms_per_min"]
            )

        else:
            raise RuntimeError(f"Unknown strategy: {strategy}")

        rows.append(row)

    th_df = pd.DataFrame(rows).sort_values(
        ["selection_score", "f1", "event_detection_rate", "precision"],
        ascending=False
    ).reset_index(drop=True)

    best = th_df.iloc[0].to_dict()
    return best, th_df

In [ ]:
def plot_training_history(history_df: pd.DataFrame, out_dir: Path):
    if len(history_df) == 0:
        return

    # train loss
    save_line(
        x=history_df["epoch"].values,
        y=history_df["train_loss"].values,
        title="Training loss by epoch",
        xlabel="epoch",
        ylabel="loss",
        out_path=out_dir / "train_loss_curve.png",
        label="train_loss"
    )

    # val PR-AUC / F1
    save_multi_line(
        curves=[
            {"x": history_df["epoch"].values, "y": history_df["val_pr_auc"].values, "label": "val_pr_auc"},
            {"x": history_df["epoch"].values, "y": history_df["val_f1"].values, "label": "val_f1"}
        ],
        title="Validation PR-AUC and F1 by epoch",
        xlabel="epoch",
        ylabel="score",
        out_path=out_dir / "val_pr_auc_f1_curve.png"
    )


def plot_pr_curve(pred_df: pd.DataFrame, out_dir: Path, split_name: str):
    y_true = pred_df["target"].astype(int).values
    probs = pred_df["prob"].astype(float).values

    precision, recall, _ = precision_recall_curve(y_true, probs)

    plt.figure(figsize=(8, 6), dpi=120)
    plt.plot(recall, precision, linewidth=2.0)
    finalize_plot(
        title=f"Precision-Recall curve ({split_name})",
        xlabel="recall",
        ylabel="precision",
        out_path=out_dir / f"pr_curve_{split_name}.png",
        show=True
    )


def plot_threshold_search(th_df: pd.DataFrame, out_dir: Path, split_name: str):
    save_multi_line(
        curves=[
            {"x": th_df["threshold"].values, "y": th_df["precision"].values, "label": "precision"},
            {"x": th_df["threshold"].values, "y": th_df["recall"].values, "label": "recall"},
            {"x": th_df["threshold"].values, "y": th_df["f1"].values, "label": "f1"}
        ],
        title=f"Threshold search curves ({split_name})",
        xlabel="threshold",
        ylabel="score",
        out_path=out_dir / f"threshold_search_{split_name}.png"
    )


def plot_lead_time_hist(event_eval_df: pd.DataFrame, out_dir: Path):
    detected = event_eval_df[event_eval_df["detected"] == 1]["lead_time_sec"].dropna().values
    if len(detected) == 0:
        return

    save_hist(
        values=detected,
        title="Lead time distribution on detected smoke events",
        xlabel="lead time (seconds)",
        ylabel="count",
        out_path=out_dir / "lead_time_hist.png",
        bins=30
    )

def build_model(model_type: str, embedding_dim: int):
    if model_type == "embedding_only":
        return EmbeddingOnlyTCN(
            emb_dim=embedding_dim,
            hidden_dim=192,
            dropout=DROPOUT
        ).to(DEVICE)

    elif model_type == "physics_only":
        return PhysicsOnlyTCN(
            phys_dim=PHYS_DIM,
            hidden_dim=64,
            dropout=DROPOUT
        ).to(DEVICE)

    elif model_type == "fusion":
        return FusionTCN(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        ).to(DEVICE)

    elif model_type == "fusion_gated":
        return GatedFusionTCN(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            dropout=DROPOUT,
        ).to(DEVICE)

    elif model_type == "fusion_meanpool":
        return FusionMeanPool(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        ).to(DEVICE)

    elif model_type == "fusion_gru":
        return FusionGRU(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        ).to(DEVICE)

    else:
        raise RuntimeError(f"Unknown model_type : {model_type}")

In [ ]:
# one wrapper to train/evaluate/save a single experiment

def run_experiment(
    exp_name: str,
    run_name: str,
    backbone_name: str,
    model_type: str,
    epochs: int,
    use_physics: bool,
):
    print(f"\n{'='*80}")
    print("Running experiment :", exp_name)
    print(f"{'='*80}")

    run_dir = ART_INDICES / run_name
    assert run_dir.exists(), f"Missing run folder : {run_dir}"

    cfg = json.loads((run_dir / "config.json").read_text())
    local_clips_df = pd.read_csv(run_dir / "clips_index.csv")

    if DEBUG_SMALL_RUN:
        local_clips_df = (
            local_clips_df.groupby("split", group_keys=False)
            .apply(lambda x: x.sample(min(len(x), DEBUG_MAX_ROWS_PER_SPLIT), random_state=SEED))
            .reset_index(drop=True)
        )

    exp_result_root = ensure_dir(ART_RESULTS / run_name)
    exp_model_root = ensure_dir(ART_MODELS / run_name)
    
    out_dir = ensure_dir(exp_result_root / exp_name)
    ckpt_dir = ensure_dir(exp_model_root / exp_name)

    train_df, val_df, test_df, train_loader, val_loader, test_loader, embedding_dim = make_loaders(
        clips_df=local_clips_df,
        videos_df=videos_df,
        backbone_name=backbone_name,
        use_physics=use_physics,
        batch_size=BATCH_SIZE
    )

    val_video_inventory = videos_df[videos_df["video_id"].isin(val_df["video_id"].unique())].copy()
    val_events = events_df[events_df["video_id"].isin(val_df["video_id"].unique())].copy()

    # model selection
    model = build_model(model_type=model_type, embedding_dim=embedding_dim)

    pos_weight_val = get_pos_weight(train_df)
    pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history_rows = []
    best_val_score = -1.0
    best_ckpt_path = ckpt_dir / "best_checkpoint.pt"

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_pred_df = predict_loader(model, val_loader, DEVICE)

        best_val_threshold, val_threshold_df = find_best_threshold(val_pred_df, THRESHOLDS)
        val_metrics = binary_metrics_from_df(val_pred_df, best_val_threshold["threshold"])

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_threshold": float(best_val_threshold["threshold"]),
            "val_pr_auc": float(val_metrics["pr_auc"]),
            "val_roc_auc": float(val_metrics["roc_auc"]),
            "val_precision": float(val_metrics["precision"]),
            "val_recall": float(val_metrics["recall"]),
            "val_f1": float(val_metrics["f1"])
        }
        history_rows.append(row)

        print(pd.DataFrame([row]))

        if val_metrics["f1"] > best_val_score:
            best_val_score = float(val_metrics["f1"])

            torch.save({
                "model_state_dict": model.state_dict(),
                "model_type": model_type,
                "backbone_name": backbone_name,
                "run_name": run_name,
                "use_physics": use_physics,
                "embedding_dim": embedding_dim,
                "threshold": float(best_val_threshold["threshold"]),
                "epoch": epoch,
                "selection_metric": "f1",
                "selection_metric_value": float(val_metrics["f1"])
            }, best_ckpt_path)

            print("Saved new best checkpoint :", best_ckpt_path)

    history_df = pd.DataFrame(history_rows)
    save_df(history_df, out_dir / "epoch_history.csv")
    plot_training_history(history_df, out_dir)

    # reload best
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)

    best_model = build_model(model_type=model_type, embedding_dim=embedding_dim)

    best_model.load_state_dict(checkpoint["model_state_dict"])

    # val predictions using best model
    val_pred_df = predict_loader(best_model, val_loader, DEVICE)
    best_val_threshold, val_threshold_df = find_best_threshold_with_event_metrics(
        pred_df=val_pred_df,
        events_df=val_events,
        video_inventory_df=val_video_inventory,
        thresholds=THRESHOLDS,
        w_frames=int(cfg["w_frames"]),
        fps=float(cfg["fps"]),
        strategy="f1_event_balanced",
        max_false_alarms_per_min=None
    )
    
    best_threshold = float(best_val_threshold["threshold"])

    save_df(val_pred_df, out_dir / "val_predictions.csv")
    save_df(val_threshold_df, out_dir / "val_threshold_search.csv")
    plot_threshold_search(val_threshold_df, out_dir, split_name="val")
    plot_pr_curve(val_pred_df, out_dir, split_name="val")

    # test predictions
    test_pred_df = predict_loader(best_model, test_loader, DEVICE)
    save_df(test_pred_df, out_dir / "test_predictions.csv")
    plot_pr_curve(test_pred_df, out_dir, split_name="test")

    # clip-level test metrics
    test_metrics = binary_metrics_from_df(test_pred_df, best_threshold)

    # event-level test metrics
    test_video_inventory = videos_df[videos_df["video_id"].isin(test_df["video_id"].unique())].copy()
    test_events = events_df[events_df["video_id"].isin(test_df["video_id"].unique())].copy()

    event_metrics, event_eval_df, stability_df = compute_event_metrics(
        pred_df=test_pred_df,
        events_df=test_events,
        video_inventory_df=test_video_inventory,
        threshold=best_threshold,
        w_frames=int(cfg["w_frames"]),
        fps=float(cfg["fps"]),
    )

    save_df(event_eval_df, out_dir / "event_eval_test.csv")
    save_df(stability_df, out_dir / "clip_prediction_stability_test.csv")
    plot_lead_time_hist(event_eval_df, out_dir)

    # per-video false positive summary
    test_pred_df["pred"] = (test_pred_df["prob"] >= best_threshold).astype(int)
    fp_video_df = (
        test_pred_df[(test_pred_df["pred"] == 1) & (test_pred_df["target"] == 0)]
        .groupby("video_id")
        .size()
        .reset_index(name="false_positive_clips")
        .sort_values("false_positive_clips", ascending=False)
    )
    save_df(fp_video_df, out_dir / "false_positive_clips_by_video.csv")

    if len(fp_video_df) > 0:
        top_fp = fp_video_df.head(15)
        save_bar(
            categories=top_fp["video_id"].tolist(),
            values=top_fp["false_positive_clips"].tolist(),
            title="False positive clips by video (top 15)",
            xlabel="video_id",
            ylabel="false positive clips",
            out_path=out_dir / "false_positive_clips_by_video_top15.png",
            rotate=45,
            show_values=True
        )

    # inference latency estimate
    best_model.eval()
    latency_times = []

    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            x_emb = batch["x_emb"].to(DEVICE)
            x_phys = batch["x_phys"].to(DEVICE)

            t0 = time.time()
            logits, _ = best_model(x_emb, x_phys)
            _ = torch.sigmoid(logits)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            dt = time.time() - t0
            latency_times.append(dt / len(x_emb))

            if i >= 20:
                break

    latency_per_clip_sec = float(np.mean(latency_times)) if len(latency_times) > 0 else float("nan")

    summary = {
        "exp_name": exp_name,
        "run_name": run_name,
        "backbone_name": backbone_name,
        "model_type": model_type,
        "use_physics": use_physics,
        "epochs": epochs,
        "best_threshold": best_threshold,
        "params_trainable": count_parameters(best_model),
        "latency_per_clip_sec": latency_per_clip_sec,
        "clip_metrics_test": test_metrics,
        "event_metrics_test": event_metrics
    }

    save_json(summary, out_dir / "summary.json")

    flat_summary = {
        "exp_name": exp_name,
        "run_name": run_name,
        "backbone_name": backbone_name,
        "model_type": model_type,
        "use_physics": use_physics,
        "epochs": epochs,
        "best_threshold": best_threshold,
        "params_trainable": count_parameters(best_model),
        "latency_per_clip_sec": latency_per_clip_sec,
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "event_detection_rate": event_metrics["event_detection_rate"],
        "missed_detection_rate": event_metrics["missed_detection_rate"],
        "mean_lead_time_sec": event_metrics["mean_lead_time_sec"],
        "median_lead_time_sec": event_metrics["median_lead_time_sec"],
        "false_alarms_per_min": event_metrics["false_alarms_per_min"],
        "clip_prediction_stability": event_metrics["clip_prediction_stability"]
    }

    print("\nExperiment summary:")
    display(pd.DataFrame([flat_summary]))

    return flat_summary

## Stage 6 - Experiments and ablations

In [ ]:
# experiment plans

# PLAN A - horizon comparison using EfficientNet baseline
HORIZON_EXPERIMENTS = [
    {
        "exp_name": f"horizon_{run_name}_effb0_baseline",
        "run_name": run_name,
        "backbone_name": "effb0",
        "model_type": "embedding_only",
        "epochs": EPOCHS_BASELINE,
        "use_physics": False,
    }
    for run_name in HORIZON_RUNS
]

# PLAN B - backbone comparison on the chosen run
BACKBONE_EXPERIMENTS = [
    {
        "exp_name": f"backbone_{RUN_NAME}_effb0_baseline",
        "run_name": RUN_NAME,
        "backbone_name": "effb0",
        "model_type": "embedding_only",
        "epochs": EPOCHS_BASELINE,
        "use_physics": False,
    },
    {
        "exp_name": f"backbone_{RUN_NAME}_convnext_baseline",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "embedding_only",
        "epochs": EPOCHS_BASELINE,
        "use_physics": False,
    }
]

# PLAN C - main ablations on the chosen run/backbone
MAIN_ABLATIONS = [
    {
        "exp_name": f"abl_{RUN_NAME}_embedding_only",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "embedding_only",
        "epochs": EPOCHS_BASELINE,
        "use_physics": False
    },
    {
        "exp_name": f"abl_{RUN_NAME}_physics_only",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "physics_only",
        "epochs": EPOCHS_BASELINE,
        "use_physics": True
    },
    {
        "exp_name": f"abl_{RUN_NAME}_fusion",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "fusion",
        "epochs": EPOCHS_FUSION,
        "use_physics": True
    },
    {
        "exp_name": f"abl_{RUN_NAME}_fusion_gated",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "fusion_gated",
        "epochs": EPOCHS_FUSION,
        "use_physics": True
    }
]

# PLAN D - temporal model comparison on the chosen run/backbone
TEMPORAL_COMPARISON_EXPERIMENTS = [
    {
        "exp_name": f"temp_{RUN_NAME}_fusion_meanpool",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "fusion_meanpool",
        "epochs": EPOCHS_FUSION,
        "use_physics": True
    },
    {
        "exp_name": f"temp_{RUN_NAME}_fusion_gru",
        "run_name": RUN_NAME,
        "backbone_name": "convnext_tiny",
        "model_type": "fusion_gru",
        "epochs": EPOCHS_FUSION,
        "use_physics": True
    }
]

print("Horizon experiments :", len(HORIZON_EXPERIMENTS))
print("Backbone experiments:", len(BACKBONE_EXPERIMENTS))
print("Main ablations      :", len(MAIN_ABLATIONS))
print("Temporal comparison :", len(TEMPORAL_COMPARISON_EXPERIMENTS))

In [ ]:
# choose which experiment group to run (comment and uncommment as needed)
#EXPERIMENTS_TO_RUN = HORIZON_EXPERIMENTS
#EXPERIMENTS_TO_RUN = BACKBONE_EXPERIMENTS
EXPERIMENTS_TO_RUN = MAIN_ABLATIONS
#EXPERIMENTS_TO_RUN = TEMPORAL_COMPARISON_EXPERIMENTS

all_results = []

for spec in EXPERIMENTS_TO_RUN:
    out = run_experiment(
        exp_name=spec["exp_name"],
        run_name=spec["run_name"],
        backbone_name=spec["backbone_name"],
        model_type=spec["model_type"],
        epochs=spec["epochs"],
        use_physics=spec["use_physics"]
    )
    all_results.append(out)

results_df = pd.DataFrame(all_results)
display(results_df)

COMPARE_DIR = ensure_dir(ART_RESULTS / "comparisons")
save_df(results_df, COMPARE_DIR / "latest_experiment_results.csv")
save_df(results_df, COMPARE_DIR / f"results_{int(time.time())}.csv")

print("Saved latest results to :", COMPARE_DIR / "latest_experiment_results.csv")

In [ ]:
# compare latest batch of experiments with a few summary plots

latest_results_path = ART_RESULTS / "comparisons" / "latest_experiment_results.csv"
assert latest_results_path.exists(), f"Missing file : {latest_results_path}"

latest_results_df = pd.read_csv(latest_results_path)
display(latest_results_df)

# plot 1 - PR-AUC
save_bar(
    categories=latest_results_df["exp_name"].tolist(),
    values=latest_results_df["test_pr_auc"].round(4).tolist(),
    title="Test PR-AUC by experiment",
    xlabel="experiment",
    ylabel="PR-AUC",
    out_path=RUN_RESULT_DIR / "compare_test_pr_auc.png",
    rotate=45,
    show_values=True
)

# plot 2 - F1
save_bar(
    categories=latest_results_df["exp_name"].tolist(),
    values=latest_results_df["test_f1"].round(4).tolist(),
    title="Test F1 by experiment",
    xlabel="experiment",
    ylabel="F1",
    out_path=RUN_RESULT_DIR / "compare_test_f1.png",
    rotate=45,
    show_values=True
)

# plot 3 - event detection rate
save_bar(
    categories=latest_results_df["exp_name"].tolist(),
    values=latest_results_df["event_detection_rate"].round(4).tolist(),
    title="Event detection rate by experiment",
    xlabel="experiment",
    ylabel="event detection rate",
    out_path=RUN_RESULT_DIR / "compare_event_detection_rate.png",
    rotate=45,
    show_values=True
)

# plot 4 - mean lead time
save_bar(
    categories=latest_results_df["exp_name"].tolist(),
    values=latest_results_df["mean_lead_time_sec"].fillna(0).round(4).tolist(),
    title="Mean lead time by experiment",
    xlabel="experiment",
    ylabel="lead time (seconds)",
    out_path=RUN_RESULT_DIR / "compare_mean_lead_time.png",
    rotate=45,
    show_values=True
)

# plot 5 - false alarms per min
save_bar(
    categories=latest_results_df["exp_name"].tolist(),
    values=latest_results_df["false_alarms_per_min"].round(4).tolist(),
    title="False alarms per minute by experiment",
    xlabel="experiment",
    ylabel="false alarms / min",
    out_path=RUN_RESULT_DIR / "compare_false_alarms_per_min.png",
    rotate=45,
    show_values=True
)

## Stage 7 - Final fusion evaluation and qualitative checks

In [ ]:
# load the final fusion run for inspection
FINAL_EXPERIMENT_NAME = f"abl_{RUN_NAME}_fusion"

FINAL_EXP_DIR = RUN_RESULT_DIR / FINAL_EXPERIMENT_NAME
FINAL_CKPT_DIR = RUN_MODEL_DIR / FINAL_EXPERIMENT_NAME

assert FINAL_EXP_DIR.exists(), f"Missing experiment folder : {FINAL_EXP_DIR}"
assert FINAL_CKPT_DIR.exists(), f"Missing checkpoint folder : {FINAL_CKPT_DIR}"

final_summary = json.loads((FINAL_EXP_DIR / "summary.json").read_text())
display(pd.DataFrame([{
    **{k: v for k, v in final_summary.items() if k not in ["clip_metrics_test", "event_metrics_test"]},
    **final_summary["clip_metrics_test"],
    **final_summary["event_metrics_test"]
}]))

In [ ]:
def build_model_from_checkpoint(checkpoint):
    model_type = checkpoint["model_type"]
    backbone_name = checkpoint["backbone_name"]
    embedding_dim = int(checkpoint["embedding_dim"])

    if model_type == "embedding_only":
        model = EmbeddingOnlyTCN(
            emb_dim=embedding_dim,
            hidden_dim=192,
            dropout=DROPOUT
        )
    elif model_type == "physics_only":
        model = PhysicsOnlyTCN(
            phys_dim=PHYS_DIM,
            hidden_dim=64,
            dropout=DROPOUT
        )
    elif model_type == "fusion":
        model = FusionTCN(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        )

    elif model_type == "fusion_gated":
        model = GatedFusionTCN(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            dropout=DROPOUT,
        )

    elif model_type == "fusion_meanpool":
        model = FusionMeanPool(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        )

    elif model_type == "fusion_gru":
        model = FusionGRU(
            emb_dim=embedding_dim,
            phys_dim=PHYS_DIM,
            hidden_dim=192,
            phys_hidden=48,
            dropout=DROPOUT,
        )
        
    else:
        raise RuntimeError(f"Unknown model_type in checkpoint : {model_type}")

    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()

    return model

final_ckpt_path = FINAL_CKPT_DIR / "best_checkpoint.pt"
assert final_ckpt_path.exists(), f"Missing checkpoint : {final_ckpt_path}"

final_ckpt = torch.load(final_ckpt_path, map_location=DEVICE)
final_model = build_model_from_checkpoint(final_ckpt)
final_threshold = float(final_ckpt["threshold"])

print("Loaded final model.")
print("Threshold :", final_threshold)
print("Params    :", count_parameters(final_model))

In [ ]:
def get_backbone_dir_and_dim(backbone_name):
    if backbone_name == "effb0":
        return EFF_DIR, int(EFF_META["embedding_dim"])
    elif backbone_name == "convnext_tiny":
        return CONVNEXT_DIR, int(CONVNEXT_META["embedding_dim"])
    else:
        raise RuntimeError(f"Unknown backbone : {backbone_name}")

def predict_single_video_timeline(video_id: str, run_name: str, model, threshold: float, backbone_name: str):
    run_dir = ART_INDICES / run_name
    cfg = json.loads((run_dir / "config.json").read_text())
    clips_df_local = pd.read_csv(run_dir / "clips_index.csv")

    backbone_dir, embedding_dim = get_backbone_dir_and_dim(backbone_name)

    video_clips = clips_df_local[clips_df_local["video_id"] == video_id].copy().sort_values("t").reset_index(drop=True)
    assert len(video_clips) > 0, f"No clips found for video : {video_id}"

    vid_emb = np.load(backbone_dir / f"{video_id}.npy", mmap_mode="r")
    vid_phys = np.load(PHYSICS_DIR / f"{video_id}.npy", mmap_mode="r")

    rows = []

    with torch.no_grad():
        for r in tqdm(video_clips.itertuples(index=False), total=len(video_clips), desc=f"Predicting {video_id}"):
            s = int(r.clip_start)
            e = int(r.clip_end)

            x_emb = np.asarray(vid_emb[s:e+1], dtype=np.float32)
            x_phys = np.asarray(vid_phys[s:e+1], dtype=np.float32)
            x_phys = (x_phys - phys_mean) / phys_std

            x_emb = torch.from_numpy(x_emb).unsqueeze(0).to(DEVICE)
            x_phys = torch.from_numpy(x_phys).unsqueeze(0).to(DEVICE)

            logits, attn = model(x_emb, x_phys)
            prob = float(torch.sigmoid(logits).cpu().item())

            rows.append({
                "video_id": video_id,
                "t": int(r.t),
                "clip_start": s,
                "clip_end": e,
                "target": int(r.target),
                "prob": prob,
                "pred": int(prob >= threshold)
            })

    return pd.DataFrame(rows)


# choose one test video
example_test_video = split_info["videos"]["test"][0]
video_pred_df = predict_single_video_timeline(
    video_id=example_test_video,
    run_name=RUN_NAME,
    model=final_model,
    threshold=final_threshold,
    backbone_name=final_ckpt["backbone_name"]
)

save_df(video_pred_df, FINAL_EXP_DIR / f"{example_test_video}_timeline_predictions.csv")
display(video_pred_df.head(10))

In [ ]:
# qualitative timeline plot for one example test video

video_events = events_df[events_df["video_id"] == example_test_video].copy().sort_values("event_start")

plt.figure(figsize=(14, 5), dpi=120)
plt.plot(video_pred_df["t"] / run_fps, video_pred_df["prob"], linewidth=1.8, label="predicted smoke-onset probability")
plt.axhline(final_threshold, linestyle="--", linewidth=1.5, label=f"threshold = {final_threshold:.2f}")

for i, ev in enumerate(video_events.itertuples(index=False)):
    x = ev.event_start / run_fps
    plt.axvline(x, linestyle=":", linewidth=1.2, label="smoke event start" if i == 0 else None)

plt.legend()
finalize_plot(
    title=f"Prediction timeline for {example_test_video}",
    xlabel="time (seconds)",
    ylabel="predicted probability",
    out_path=FINAL_EXP_DIR / f"{example_test_video}_timeline_plot.png",
    show=True
)

In [ ]:
# aggregate prediction curves relative to event start

def build_event_aligned_curve(pred_df: pd.DataFrame, events_df_local: pd.DataFrame, fps: float, w_frames: int):
    curves = []

    pred_by_video = {
        vid: g.sort_values("t").reset_index(drop=True)
        for vid, g in pred_df.groupby("video_id")
    }

    for r in events_df_local.itertuples(index=False):
        vid = r.video_id
        event_start = int(r.event_start)

        if vid not in pred_by_video:
            continue

        g = pred_by_video[vid].copy()
        g["delta_frames"] = event_start - g["t"]
        g = g[(g["delta_frames"] >= 0) & (g["delta_frames"] <= w_frames)].copy()

        if len(g) == 0:
            continue

        g["delta_sec"] = g["delta_frames"] / fps
        curves.append(g[["delta_sec", "prob"]].copy())

    if len(curves) == 0:
        return None

    # bin by 0.1s
    all_df = pd.concat(curves, ignore_index=True)
    all_df["delta_bin"] = (all_df["delta_sec"] / 0.1).round().astype(int) * 0.1

    mean_curve = (
        all_df.groupby("delta_bin")["prob"]
        .mean()
        .reset_index()
        .sort_values("delta_bin", ascending=False)
    )

    return mean_curve


test_pred_path = FINAL_EXP_DIR / "test_predictions.csv"
assert test_pred_path.exists(), f"Missing file : {test_pred_path}"

test_pred_df = pd.read_csv(test_pred_path)
test_events_df = events_df[events_df["video_id"].isin(split_info["videos"]["test"])].copy()

mean_curve_df = build_event_aligned_curve(
    pred_df=test_pred_df,
    events_df_local=test_events_df,
    fps=run_fps,
    w_frames=run_w_frames
)

if mean_curve_df is not None:
    plt.figure(figsize=(10, 6), dpi=120)
    plt.plot(mean_curve_df["delta_bin"], mean_curve_df["prob"], linewidth=2.0)
    finalize_plot(
        title="Mean prediction probability before smoke onset",
        xlabel="seconds before onset",
        ylabel="mean predicted probability",
        out_path=FINAL_EXP_DIR / "event_aligned_probability_curve.png",
        show=True
    )
    save_df(mean_curve_df, FINAL_EXP_DIR / "event_aligned_probability_curve.csv")
else:
    print("Could not build event-aligned curve.")